In [ ]:
import os
from os.path import expanduser
home = expanduser("~/")

import sys
# sys.path.insert(0, '/global/u2/x/xshuang/gigalens-xh-dev/src')

# import sys
conda_env = sys.path[1]
del sys.path[1]

import os
# sys.path.append(f'{os.environ['HOME']}/gigalens_personal/gigalens/src')
sys.path.append(home+'/gigalens'+'/src')
sys.path.append(conda_env)
sys.path.append(home+'/GIGALens-Code/')
print(sys.path)



srcdir = os.path.join(home, "gigalens/src/")


In [ ]:
import tensorflow_probability.substrates.jax as tfp

from gigalens.jax.inference import ModellingSequence
from gigalens.jax.model import ForwardProbModel, BackwardProbModel
from gigalens.model import PhysicalModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic
from gigalens.jax.profiles.mass import epl, shear

import jax
from jax import random
import numpy as np
import optax
from jax import numpy as jnp
from matplotlib import pyplot as plt
import optax
import corner
import yaml
import pickle
# from helpers import *
import blackjax
import importlib
tfd = tfp.distributions

from mclmc_alt import MCLMC

In [ ]:
#* Define Priors
lens_prior = tfd.JointDistributionSequential(
    [
        tfd.JointDistributionNamed(
            dict(
                theta_E=tfd.LogNormal(jnp.log(1.25), 0.25),
                gamma=tfd.TruncatedNormal(2, 0.25, 1, 3),
                e1=tfd.Normal(0, 0.1),
                e2=tfd.Normal(0, 0.1),
                center_x=tfd.Normal(0, 0.05),
                center_y=tfd.Normal(0, 0.05),
            )
        ),
        tfd.JointDistributionNamed(
            dict(gamma1=tfd.Normal(0, 0.05), gamma2=tfd.Normal(0, 0.05))
        ),
    ]
)
lens_light_prior = tfd.JointDistributionSequential(
    [
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(1.0), 0.15),
                n_sersic=tfd.Uniform(2, 6),
                e1=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
                e2=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
                center_x=tfd.Normal(0, 0.05),
                center_y=tfd.Normal(0, 0.05),
                Ie=tfd.LogNormal(jnp.log(500.0), 0.3),
            )
        )
    ]
)

source_light_prior = tfd.JointDistributionSequential(
    [
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(0.25), 0.15),
                n_sersic=tfd.Uniform(0.5, 4),
                e1=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5),
                e2=tfd.TruncatedNormal(0, 0.15, -0.5, 0.5),
                center_x=tfd.Normal(0, 0.25),
                center_y=tfd.Normal(0, 0.25),
                Ie=tfd.LogNormal(jnp.log(150.0), 0.5),
            )
        )
    ]
)

prior = tfd.JointDistributionSequential(
    [lens_prior, lens_light_prior, source_light_prior]
)

In [ ]:
gigal_dir = os.path.join(home,'gigalens/src/gigalens/')
kernel = np.load(gigal_dir + '/assets/psf.npy').astype(np.float32)
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=60, supersample=2, kernel=kernel)
phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False)])
lens_sim = LensSimulator(phys_model, sim_config, bs=1)
observed_img = np.load(gigal_dir + '/assets/demo.npy')
prob_model = ForwardProbModel(prior, observed_img, background_rms=0.2, exp_time=100)
model_seq = ModellingSequence(phys_model, prob_model, sim_config)

In [ ]:
opt = optax.adabelief(1e-2, b1=0.95, b2=0.99)
best, chisq, lp = model_seq.MAP(opt, seed=0)

In [ ]:
opt = optax.adabelief(1e-4, b1=0.95, b2=0.99)
qz, loss_hist = model_seq.SVI(best, opt, n_vi=1000, num_steps=1500)

In [ ]:
hmc_samples = model_seq.HMC(qz, num_burnin_steps=250, num_results=750, n_hmc=16, pbar_interval=25)

In [ ]:
# Note that right now, MCLMC only runs on 1 GPU, so times are not comparable
mclmc_samples = MCLMC(model_seq, qz, n_hmc=16, num_burnin_steps=1000, num_results=2000)

### Calculate Performance Metrics for HMC and MCLMC
Note that this is not an even comparison. The number of gradient evaluations HMC does depends on what happens with the leapfrog adaptation. In contrast, MCLMC does however many gradient evaluations are needed to do one integration step (with the default integrator, isokinetic mclachlan, that's 2).

Because of this, there's no nice ratio to compare number of HMC steps to MLCMC steps. You have to add a callback into HMC to count the number of gradient evaluations it does. That being said, this should give you a rough idea of the algorithms' relative performance.

In [ ]:
ESS_hmc = blackjax.diagnostics.effective_sample_size(hmc_samples, chain_axis=1, sample_axis=0)
ESS_mclmc = blackjax.diagnostics.effective_sample_size(mclmc_samples, chain_axis=1, sample_axis=0)

print("HMC: Mean ESS | Min ESS")
print(np.mean(ESS_hmc), " | ", jnp.min(ESS_hmc))
print("MCLMC: Mean ESS | Min ESS")
print(np.mean(ESS_mclmc), " | ", jnp.min(ESS_mclmc))

In [ ]:
Rhat_hmc = blackjax.diagnostics.potential_scale_reduction(hmc_samples, chain_axis=1, sample_axis=0)
Rhat_mclmc = blackjax.diagnostics.potential_scale_reduction(mclmc_samples, chain_axis=1, sample_axis=0)

print("HMC: Mean Rhat | Max Rhat")
print(np.mean(Rhat_hmc), " | ", jnp.max(Rhat_hmc))
print("MCLMC: Mean Rhat | Max Rhat")
print(np.mean(Rhat_mclmc), " | ", jnp.max(Rhat_mclmc))

### More involved demo 
Here, you can see a bit more of the inner workings. Should essentially do the same thing though.

In [ ]:


def log_prob(z):
    return prob_model.log_prob(lens_sim, z)[0]

# start = jnp.squeeze(results['SVI'].qz.mean())
inv_mass_mat = qz.covariance()

inv_mass_mat_true = jnp.cov(hmc_samples.reshape(-1, 22).T)

transform = lambda state, info: state.position


In [ ]:
from mclmc_alt import isokinetic_mclachlan_smart, mclachlan_coefficients, mclmc_find_L_and_step_size_smart, MCLMCAdaptationState

n_grad_per_integration_step_mclachlan = len(mclachlan_coefficients)//2 #* Happens on every odd step of the integrator

integrator = isokinetic_mclachlan_smart

 # build the kernel
kernel = lambda inverse_mass_matrix : blackjax.mcmc.mclmc.build_kernel(
    logdensity_fn=log_prob,
    integrator=integrator,
    inverse_mass_matrix=inverse_mass_matrix,
)

dim =22

desired_energy_variance= 5e-4

In [ ]:
from mclmc_alt import isokinetic_mclachlan_smart, mclmc_find_L_and_step_size_smart, MCLMCAdaptationState, init_multi, mclmc_multi


rng_key = jax.random.key(0)
init_key, tune_key, run_key = jax.random.split(rng_key, 3)

# start_loc = jnp.squeeze(results['SVI'].qz.sample(1, seed=init_key))

# initial_state = blackjax.mcmc.mclmc.init(
#     position=start_loc, logdensity_fn=log_prob, rng_key=init_key
# )

n_chains = 4
state_multi = init_multi(qz.sample((n_chains,), seed=init_key), init_key, log_prob)

# blackjax_state_after_tuning, blackjax_mclmc_sampler_params = burnin(tune_key, initial_state, steps=10000)
starting_adapt_state = blackjax.adaptation.mclmc_adaptation.MCLMCAdaptationState(
    jnp.sqrt(dim), jnp.sqrt(dim) * 0.25, inverse_mass_matrix=inv_mass_mat
)

starttime = time.perf_counter()
# find values for L and step_size
(
    blackjax_state_after_tuning,
    blackjax_mclmc_sampler_params,
    _
) = mclmc_find_L_and_step_size_smart(
    mclmc_kernel=kernel,
    num_steps=1000,
    state=state_multi,
    rng_key=tune_key,
    frac_tune1=0.1, #* initial step size tuning
    frac_tune2=0.7, #* Used for mass matrix adaptation
    frac_tune3=0.2, #! Tuning L. ~10 effective samples are needed for this to be accurate
    params=starting_adapt_state,
    desired_energy_var=desired_energy_variance,
    multi_chain=True,
    num_chains=n_chains,
    mass_matrix_adapt=False
)

total_time = time.perf_counter()-starttime
print("Burnin Time:", total_time)

L = blackjax_mclmc_sampler_params.L
step_size = blackjax_mclmc_sampler_params.step_size
print(f"ADAPTED. L: {L}, step_size: {step_size}, L/step: {L/step_size}")


sampling_alg = mclmc_multi(
    log_prob,
    L=L,
    step_size=step_size,
    num_chains=n_chains,
    inverse_mass_matrix=blackjax_mclmc_sampler_params.inverse_mass_matrix,
    integrator=integrator,
)

starttime = time.perf_counter()
_, multi_chain_samples = blackjax.util.run_inference_algorithm(
    rng_key=run_key,
    initial_state=blackjax_state_after_tuning,
    inference_algorithm=sampling_alg,
    num_steps=2000,
    transform=transform,
    progress_bar=True,
)

multi_chain_samples = jnp.transpose(multi_chain_samples, axes=(1, 0, 2))

total_time = time.perf_counter()-starttime
print(f"Sampling took {total_time} s")

In [ ]:
# n_grad_evals_mclmc = num_steps
n_grad_evals_mclmc = multi_chain_samples.shape[0]*multi_chain_samples.shape[1] * n_grad_per_integration_step_mclachlan
# ESS_mclmc = blackjax.diagnostics.effective_sample_size(samples_mclmc[np.newaxis, :,:], chain_axis=0, sample_axis=1)
ESS_mclmc = blackjax.diagnostics.effective_sample_size(multi_chain_samples, chain_axis=0, sample_axis=1)

print(ESS_mclmc)
print(np.mean(ESS_mclmc)/n_grad_evals_mclmc, " | ", jnp.min(ESS_mclmc)/n_grad_evals_mclmc)

In [ ]:
rhat = blackjax.diagnostics.potential_scale_reduction(multi_chain_samples, chain_axis=0, sample_axis=1)
print(rhat)
print(jnp.max(rhat))

In [ ]:

sample_length = list(range(20, multi_chain_samples.shape[1], 200)) + [multi_chain_samples.shape[1]]
rhats = -np.ones((len(sample_length), multi_chain_samples.shape[-1]))
for i, l in enumerate(sample_length):
    rhats[i] = blackjax.diagnostics.potential_scale_reduction(multi_chain_samples[:, :l, :], chain_axis=0, sample_axis=1)
plt.axhline(1e-2, linestyle='--', label='Convergence')
plt.plot(sample_length, rhats-1)
plt.yscale('log')
plt.ylabel("Rhat-1")
plt.xlabel("num_results")
plt.legend()
plt.show()